# 00 Overview: ทำอะไรไปแล้วบ้าง

เปิดไฟล์นี้เพื่อดูสถานะของทั้งโปรเจกต์ในที่เดียว รันจากบนลงล่างได้ทุกเมื่อ

- kernel **`Python (rwd)`**
- **อ่านอย่างเดียว**: ไม่เทรน ไม่สร้างข้อมูล และ **ไม่วัดบน test** แค่อ่านไฟล์ผลที่มีอยู่แล้ว
- ตัวเลข **val** ใช้เลือกโมเดลเท่านั้น ห้ามนำไปเผยแพร่ ตัวเลขที่เผยแพร่ได้คือ **test** และต้องรายงาน recall คู่เสมอ
- งานจริงอยู่ใน notebook แยกตาม phase: `01_data_setup` (ข้อมูล), `02_train` (เทรน), `03_evaluate` (วัดผล)

## 1. Setup

In [ ]:
import os, sys, json, csv, subprocess, datetime
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
from rwd.train import run_state

RUNS = sorted(p for p in (ROOT / "runs").glob("*") if (p / "args.yaml").exists() and not p.name.endswith("_smoke"))
def load(p):
    return json.loads(p.read_text()) if p.exists() else None
def mtime(p):
    return datetime.datetime.fromtimestamp(p.stat().st_mtime).strftime("%Y-%m-%d %H:%M")
print("repo:", ROOT)
print("runs:", [p.name for p in RUNS])

## 2. สถานะแต่ละ phase

อ่านจากไฟล์บนดิสก์ ไม่ได้พิมพ์มือ

In [ ]:
ds = ROOT / "data/weedyrice_seg"
print("Phase 2 ชุดข้อมูล:", "สร้างแล้ว" if (ds / ".rwd_prepared").exists() else "ยังไม่ได้สร้าง (รัน 01_data_setup)")
for split in ("train", "val", "test"):
    d = ds / "images" / split
    if d.exists():
        print(f"  {split:<6} {sum(1 for _ in d.iterdir()):>4} ภาพ")

print("\nPhase 3 เทรน:")
for r in RUNS:
    rows = list(csv.DictReader((r / "results.csv").open())) if (r / "results.csv").exists() else []
    args = dict(l.split(": ", 1) for l in (r / "args.yaml").read_text().splitlines() if ": " in l)
    print(f"  {r.name:<34} {run_state(r):<10} epoch {len(rows):>3}  model {args.get('model', '?'):<16} batch {args.get('batch', '?')}")
training = subprocess.run(["pgrep", "-f", "rwd.train --config"], capture_output=True, text=True).stdout.split()
print("  กำลังเทรนอยู่:", "ใช่" if training else "ไม่")

print("\nPhase 4 test (ใช้ได้ครั้งเดียวต่อโมเดล):")
used = [r for r in RUNS if (r / "eval_test.json").exists()]
for r in used:
    print(f"  ใช้แล้ว: {r.name}  เมื่อ {mtime(r / 'eval_test.json')}")
if not used:
    print("  ยังไม่ได้ใช้")

## 3. เทียบโมเดลบน val

ระดับพิกเซล (ตัววัดหลัก) ที่แต่ละค่า conf และค่า instance ของ YOLO แสดงคู่กัน
val มีแค่ 41 ภาพจากวันเดียว ต่างกันเล็กน้อยถือว่าเสมอ
ถ้า run ไหนยังไม่มีผล ให้เปิด `03_evaluate.ipynb` ตั้ง `CONFIG` เป็นไฟล์ของ run นั้นแล้วรันข้อ 2

In [ ]:
vals = {r.name: load(r / "eval_val.json") for r in RUNS}
print("[val: ใช้เลือกโมเดลเท่านั้น ไม่ใช่ผลที่เผยแพร่ได้]\n")
print(f"{'run':<34} {'conf':>5} {'precision':>10} {'recall':>8} {'IoU':>7}")
for name, v in vals.items():
    if not v:
        print(f"{name:<34} ยังไม่ได้วัดบน val")
        continue
    for conf, res in v["pixel"].items():
        a = res["all"]
        print(f"{name:<34} {conf:>5} {a['precision']:>10.3f} {a['recall']:>8.3f} {a['iou']:>7.3f}")
    print()

print(f"{'run':<34} {'inst P':>7} {'inst R':>7} {'mAP50':>7} {'mAP50-95':>9}")
for name, v in vals.items():
    if v:
        i = v["instance_default_conf"]
        print(f"{name:<34} {i['metrics/precision(M)']:>7.3f} {i['metrics/recall(M)']:>7.3f} {i['metrics/mAP50(M)']:>7.3f} {i['metrics/mAP50-95(M)']:>9.3f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 5))
for name, v in vals.items():
    if not v:
        continue
    pts = [(res["all"]["recall"], res["all"]["precision"], conf) for conf, res in v["pixel"].items()]
    ax.plot([p[0] for p in pts], [p[1] for p in pts], marker="o", label=name.replace("weedyrice_", ""))
    for rec, prec, conf in pts:
        ax.annotate(conf, (rec, prec), textcoords="offset points", xytext=(4, 4), fontsize=8)
ax.set_xlabel("pixel recall (val)"); ax.set_ylabel("pixel precision (val)")
ax.set_title("val only, for model selection. Labels are conf. Up and right is better.")
ax.grid(alpha=0.3); ax.legend(); plt.show()

## 4. ผลบน test ที่วัดไปแล้ว

cell นี้แค่อ่านไฟล์ `eval_test.json` ที่มีอยู่ ไม่วัดใหม่
test คือแปลง Long Xuyen วันที่ 2025-01-15 ซึ่งไม่อยู่ใน train และ val

In [ ]:
for r in RUNS:
    t = load(r / "eval_test.json")
    if not t:
        continue
    print(f"{r.name}  วัดเมื่อ {mtime(r / 'eval_test.json')}  ({t['images']} ภาพ)")
    for conf, res in t["pixel"].items():
        print(f"  conf {conf}")
        for label, a in [("ทั้งหมด", res["all"])] + sorted(res["by_altitude"].items()):
            print(f"    {label:<8} ภาพ={a['images']:>4}  pixel precision={a['precision']:.3f}  recall={a['recall']:.3f}  IoU={a['iou']:.3f}")
    i = t["instance_default_conf"]
    print(f"  instance: precision={i['metrics/precision(M)']:.3f}  recall={i['metrics/recall(M)']:.3f}  mAP50={i['metrics/mAP50(M)']:.3f}  mAP50-95={i['metrics/mAP50-95(M)']:.3f}\n")
if not any((r / "eval_test.json").exists() for r in RUNS):
    print("ยังไม่มีผล test")

## 5. ประวัติงานล่าสุด (git)

commit ทั้งหมดอยู่ในเครื่องก่อน และ push เมื่อเจ้าของโปรเจกต์สั่งเท่านั้น

In [ ]:
print(subprocess.run(["git", "log", "--date=format:%m-%d %H:%M", "--pretty=%ad  %s", "-n", "20"], capture_output=True, text=True).stdout)
print(subprocess.run(["git", "status", "-sb"], capture_output=True, text=True).stdout)

## 6. ขั้นต่อไป

แผนและการตัดสินใจล่าสุดอยู่ใน `docs/HANDOFF.md` cell ล่างแสดงหัวข้อ "Next steps" จากไฟล์นั้น

In [ ]:
text = (ROOT / "docs/HANDOFF.md").read_text()
start = text.find("## Next steps")
end = text.find("\n## ", start + 5)
print(text[start:end if end > 0 else None] if start >= 0 else "ไม่พบหัวข้อ Next steps ใน docs/HANDOFF.md")